In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# 1. Load the dataset
df = pd.read_csv('ncr_rides_cleaned_for_dashboard.csv')

# 2. DATA AGGREGATION: Transform transaction data into time-series demand buckets
# We group by location, date, and hour, then count the number of Booking IDs
demand_df = df.groupby(
    ['Pickup Location', 'Date', 'hour', 'weekday', 'is_weekend', 'is_peak_hour', 'Event']
)['Booking ID'].count().reset_index()

# Rename the count column to our target variable 'demand'
demand_df.rename(columns={'Booking ID': 'demand'}, inplace=True)

# 3. Define Features (X) and Target (y)
features = ['Pickup Location', 'hour', 'weekday', 'is_weekend', 'is_peak_hour', 'Event']
X = demand_df[features]
y = demand_df['demand']

# 4. Train-Test Split 
# Note: For production time-series, you split chronologically. For this baseline, we use random split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Preprocessing Rules
cat_cols = ['Pickup Location', 'weekday', 'Event']
num_cols = ['hour', 'is_weekend', 'is_peak_hour']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('passthrough', 'passthrough', num_cols)
    ]
)

# 6. Build the XGBoost Regression Pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=150, 
        max_depth=7, 
        learning_rate=0.1, 
        random_state=42, 
        n_jobs=-1
    ))
])

# 7. Train the Model
model.fit(X_train, y_train)

# 8. Generate Predictions and Evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("--- DEMAND FORECASTING METRICS ---")
print(f"Average Actual Hourly Demand (Per Location): {y.mean():.2f} rides")
print(f"Mean Absolute Error (MAE):                   {mae:.2f} rides")
print(f"Root Mean Squared Error (RMSE):              {rmse:.2f} rides")

--- DEMAND FORECASTING METRICS ---
Average Actual Hourly Demand (Per Location): 1.07 rides
Mean Absolute Error (MAE):                   0.13 rides
Root Mean Squared Error (RMSE):              0.27 rides


the output is weird as we have fragmened the data into hourly metiric that to base on unique locations , theirfore Average Actual Hourly Demand (Per Location): 1.07 rides